In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect('../Data/raw/airtel.db')

def q(sql):
    return pd.read_sql(sql, con)

In [2]:
q(""" SELECT * FROM customers;""")

,customer_id,activation_date,circle_id,age_band,gender,device_type,os,acquisition_channel,connection_type,current_plan_id,arpu_inr,ab_group,is_churned,churn_date
0,1,2023-04-22,18,18-24,Male,Smartphone,iOS,Doorstep,Prepaid,4,320,B,1,2025-04-30
1,2,2023-09-06,13,35-44,Male,Smartphone,Android,Retailer,Prepaid,1,212,A,1,2025-02-11
2,3,2023-02-09,14,35-44,Female,Smartphone,Android,Doorstep,Postpaid,12,749,A,0,NaN
3,4,2024-03-16,9,45-54,Male,Feature Phone,Other,Retailer,Prepaid,9,148,B,1,2025-03-19
4,5,2022-12-10,5,35-44,Female,Smartphone,Android,Doorstep,Prepaid,8,257,B,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21995,21996,2022-04-20,1,25-34,Male,Smartphone,iOS,Retailer,Postpaid,14,1499,A,0,NaN
21996,21997,2023-05-22,14,25-34,Male,Smartphone,Android,Retailer,Prepaid,8,257,A,0,NaN
21997,21998,2023-12-29,13,18-24,Female,Feature Phone,Other,Airtel App,Prepaid,3,224,B,0,NaN
21998,21999,2024-02-16,4,45-54,Female,Smartphone,Android,Doorstep,Prepaid,6,428,A,1,2025-04-07


In [3]:
q(""" SELECT * FROM plans;""")

,plan_id,plan_name,plan_type,price_inr,validity_days,data_quota,daily_data_gb,voice,sms_per_day,ott_benefit,is_5g
0,1,Prepaid Lite 99,Prepaid,99,14,0.5 GB/day,0.50,Limited,50,NaN,0
1,2,Prepaid Smart 155,Prepaid,155,24,1 GB/day,1.00,Unlimited,100,NaN,0
2,3,Prepaid Value 209,Prepaid,209,28,1 GB/day,1.00,Unlimited,100,NaN,0
3,4,Prepaid Popular 299,Prepaid,299,28,1.5 GB/day,1.50,Unlimited,100,NaN,1
4,5,Prepaid Data 349,Prepaid,349,28,2.5 GB/day,2.50,Unlimited,100,NaN,1
5,6,Prepaid Entertainment 399,Prepaid,399,28,2.5 GB/day,2.50,Unlimited,100,OTT-Lite,1
6,7,Prepaid Max 479,Prepaid,479,56,1.5 GB/day,1.50,Unlimited,100,NaN,1
7,8,Prepaid Quarterly 719,Prepaid,719,84,2 GB/day,2.00,Unlimited,100,NaN,1
8,9,Prepaid Annual 1799,Prepaid,1799,365,2 GB/day,2.00,Unlimited,100,OTT,1
9,10,Postpaid 399,Postpaid,399,30,40 GB/month,1.33,Unlimited,100,OTT-Lite,1


In [4]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT circle_id) AS unique_circles
       FROM circles;""")

,total_rows,unique_circles
0,22,22


In [5]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT plan_id) AS unique_plans
       FROM plans;""")

,total_rows,unique_plans
0,14,14


In [6]:
q(""" SELECT COUNT(*)  AS total_rows,
       COUNT(DISTINCT customer_id) AS unique_customers
       FROM customers;""")

,total_rows,unique_customers
0,22000,22000


In [7]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT usage_id) AS unique_usages
         FROM usage_daily;""")

,total_rows,unique_usages
0,1326739,1326739


In [8]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT recharge_id) AS unique_recharges
         FROM recharges;""")

,total_rows,unique_recharges
0,408089,408089


In [9]:
q(""" SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT complaint_id) AS unique_complaints
            FROM complaints;""")

,total_rows,unique_complaints
0,26841,26841


In [10]:
q(""" SELECT COUNT(*)
FROM circles c
JOIN customers cu
ON c.circle_id = cu.circle_id;""")

,COUNT(*)
0,22000


In [11]:
q(""" SELECT COUNT(*)
FROM plans c
JOIN customers cu
ON c.plan_id = cu.current_plan_id;""")

,COUNT(*)
0,22000


In [12]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN recharges cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,408089


In [13]:
q(""" SELECT COUNT(*)
FROM plans c
JOIN recharges cu
ON c.plan_id = cu.plan_id;""")

,COUNT(*)
0,408089


In [14]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN usage_daily cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,1326739


In [15]:
q(""" SELECT COUNT(*)
FROM customers c
JOIN complaints cu
ON c.customer_id = cu.customer_id;""")

,COUNT(*)
0,26841


In [16]:
q(""" SELECT SUM(is_churned = 1 AND churn_date IS NOT NULL) AS left_dated,
       SUM(is_churned = 0 AND churn_date IS NULL) AS stayed_blank
       FROM customers;""")

,left_dated,stayed_blank
0,5226,16774


In [17]:
q(""" SELECT SUM(is_resolved = 1 AND resolution_days IS NOT NULL) AS left_dated,
       SUM(is_resolved = 0 AND resolution_days IS NULL) AS stayed_blank
       FROM complaints;""")

,left_dated,stayed_blank
0,18820,8021


In [18]:
# find the subscribers with no row in the usage_daily table

q(""" SELECT c.customer_id, c.connection_type,
       (SELECT COUNT(*)
        FROM recharges r
        WHERE r.customer_id = c.customer_id) AS recharge_count
FROM customers c
WHERE   not EXISTS (SELECT 1
                    FROM usage_daily u
                    WHERE u.customer_id = c.customer_id);""")

,customer_id,connection_type,recharge_count
0,266,Prepaid,1
1,10684,Prepaid,29
2,13508,Prepaid,8
3,19111,Prepaid,39


In [ ]:
# find each subscriber's last day of usage

q("""WITH l AS (SELECT customer_id,     
             MAX(usage_date) AS last_use         
           FROM usage_daily GROUP BY customer_id)
SELECT c.is_churned, COUNT(*) AS subs, 
       ROUND(AVG(julianday('2025-06-30')
             - julianday(l.last_use)), 2)     
         AS avg_quiet,                    
       MIN(julianday('2025-06-30')              
             - julianday(l.last_use)) AS min_,
       MAX(julianday('2025-06-30')              
             - julianday(l.last_use)) AS max_   
FROM   customers c JOIN l                
       ON l.customer_id = c.customer_id     
GROUP  BY c.is_churned;""")

,is_churned,subs,avg_quiet,min_,max_
0,0,16772,2.66,0.0,143.0
1,1,5224,87.80,7.0,180.0
